# 09 · Lista de casa: modelos pré-treinados

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- Não há ajuste fino nesta lista; a GPU ajuda, mas não é necessária.
- Cerca de metade da nota está nas respostas escritas. Use os números que você obteve.
- Antes de entregar, use **Reiniciar sessão e executar tudo**.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Um modelo pronto, medido com régua | 2,0 |
| 2 | Tokenização na prática | 2,0 |
| 3 | Um buscador semântico | 2,5 |
| 4 | Quantos exemplos são necessários? | 2,5 |
| 5 | A página de um modelo | 1,0 |

In [ ]:
import importlib.util
import subprocess
import sys

faltando = [p for p, modulo in [("transformers", "transformers"), ("sentence-transformers", "sentence_transformers")] if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)

import warnings
from pathlib import Path
from urllib.request import urlretrieve

warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from transformers import AutoTokenizer, pipeline

SEMENTE = 42
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

MODELO_SENTIMENTO = "pysentimiento/bertweet-pt-sentiment"
MODELO_BERT = "neuralmind/bert-base-portuguese-cased"
MODELO_FRASES = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False).dropna(subset=["recommend_to_a_friend", "review_text"])
df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"]
df["nao_recomenda"] = (df["recommend_to_a_friend"] == "No").astype(int)

amostra = df.sample(4000, random_state=7)[["texto", "nao_recomenda"]].reset_index(drop=True)
ajuste, valid = amostra.iloc[:3000], amostra.iloc[3000:].reset_index(drop=True)

classificador = pipeline("text-classification", model=MODELO_SENTIMENTO, device=dispositivo)
tokenizador = AutoTokenizer.from_pretrained(MODELO_BERT)
modelo_frases = SentenceTransformer(MODELO_FRASES, device=dispositivo)

print(f"dispositivo: {dispositivo} · ajuste: {len(ajuste)} · validação: {len(valid)} · rótulos do modelo de sentimento: {list(classificador.model.config.id2label.values())}")

## Questão 1 (2,0 pontos): um modelo pronto, medido com régua

O modelo de sentimento nunca viu as nossas avaliações. Vamos medir quanto ele vale como detector de "não recomenda".

a) Aplique o `classificador` aos textos de `valid`, pedindo a pontuação de **todos** os rótulos (`top_k=None`), com truncamento em 128 tokens. Guarde em `negativo_q1` um array com a pontuação do rótulo negativo (`"NEG"`) de cada avaliação.

b) Calcule `auc_q1`: a ROC AUC de `negativo_q1` contra `valid["nao_recomenda"]`.

c) Calcule `auc_tfidf_q1`: a AUC, na mesma validação, de um TF-IDF com regressão logística treinado nos 3000 exemplos de `ajuste`.

In [ ]:
# ✏️ Questão 1
# SEU CÓDIGO AQUI
negativo_q1 = ...
auc_q1 = ...
auc_tfidf_q1 = ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
assert isinstance(negativo_q1, np.ndarray) and negativo_q1.shape == (len(valid),) and ((negativo_q1 >= 0) & (negativo_q1 <= 1)).all()
assert abs(auc_q1 - roc_auc_score(valid["nao_recomenda"], negativo_q1)) < 1e-9
assert 0.9 < auc_tfidf_q1 < 1, "confira o modelo de referência: TF-IDF e regressão logística treinados em `ajuste`"
print(f"Formato correto. Modelo pronto: {auc_q1:.4f} · TF-IDF: {auc_tfidf_q1:.4f}")

**1d)** Compare as duas AUC. O modelo pronto foi treinado com tuítes, para classificar sentimento; o nosso alvo é "não recomenda", em avaliações de produtos. Cite duas diferenças entre a tarefa dele e a nossa que podem explicar o resultado. Em que situação você usaria o modelo pronto mesmo assim?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,0 pontos): tokenização na prática

a) Para cada um dos três textos abaixo, calcule o número de tokens (**sem** os tokens especiais; use `add_special_tokens=False`) dividido pelo número de palavras (separadas por espaço). Guarde em `tokens_por_palavra`, uma `Series` indexada pelo nome de cada texto.

b) Tokenize `avaliacao_longa` com truncamento em 32 tokens e converta os números de volta em texto (`tokenizador.decode`, com `skip_special_tokens=True`). Guarde em `texto_cortado_q2`.

In [ ]:
textos_q2 = {
    "formal": "O produto foi entregue dentro do prazo e corresponde à descrição do anúncio.",
    "informal": "mto bom vlw chegou rapidão amei dms super recomendo kkkk",
    "técnico": "SSD NVMe 512GB USB-C 3.2 Gen2 compatível c/ MacBook Pro M1 e Dell XPS13",
}
avaliacao_longa = valid.loc[valid["texto"].str.len().idxmax(), "texto"]
print(len(avaliacao_longa.split()), "palavras na avaliação mais longa da validação")

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
tokens_por_palavra = ...
texto_cortado_q2 = ...

In [ ]:
# ✅ Conferência da questão 2 (não altere)
assert list(tokens_por_palavra.index) == ["formal", "informal", "técnico"] and (tokens_por_palavra >= 1).all(), "cada palavra gera pelo menos um token"
_formal = len(tokenizador(textos_q2["formal"], add_special_tokens=False)["input_ids"]) / len(textos_q2["formal"].split())
assert abs(tokens_por_palavra["formal"] - _formal) < 1e-9, "não conte os tokens especiais"
assert isinstance(texto_cortado_q2, str) and 0 < len(texto_cortado_q2) < len(avaliacao_longa)
assert len(tokenizador(texto_cortado_q2, add_special_tokens=False)["input_ids"]) <= 32
print("Formato correto.")

**2c)** Qual dos três textos gera mais tokens por palavra, e por quê? Dê duas consequências práticas disso (pense em custo e no limite de tokens). No item (b), que parte da avaliação longa se perdeu, e por que isso pode ser um problema justamente neste tipo de texto?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (2,5 pontos): um buscador semântico

a) Calcule `vetores_acervo`: os vetores normalizados dos 2000 textos de `acervo`.

b) Escreva `buscar(consulta, k=5)`, que devolve um DataFrame com as colunas `posicao` (índice do texto em `acervo`), `semelhanca` e `texto`, com os `k` textos mais parecidos com a consulta, do mais para o menos parecido. Faça a conta com o NumPy.

c) Escolha **três consultas** que façam sentido para quem analisa avaliações de clientes, com palavras que provavelmente **não** aparecem nos textos (sinônimos, descrições indiretas). Guarde-as em `consultas_q3` e execute a célula de resultados.

d) Leia os cinco resultados de cada consulta e registre em `relevantes_q3` o seu julgamento: um dicionário que leva cada consulta a uma lista de cinco valores `True` ou `False` (o resultado atende à consulta?).

In [ ]:
acervo = ajuste["texto"].head(2000).tolist()

In [ ]:
# ✏️ Questão 3 (a, b, c)
# SEU CÓDIGO AQUI
vetores_acervo = ...

def buscar(consulta, k=5):
    ...

consultas_q3 = [..., ..., ...]

In [ ]:
# Resultados das suas consultas (não altere)
pd.set_option("display.max_colwidth", 140)
for consulta in consultas_q3:
    print(f"\n=== {consulta} ===")
    for linha in buscar(consulta).itertuples():
        print(f"  {linha.semelhanca:.3f}  [{linha.posicao}] {linha.texto[:130]}")

In [ ]:
# ✏️ Questão 3 (d): preencha DEPOIS de ler os resultados acima
# SEU CÓDIGO AQUI
relevantes_q3 = {
    consultas_q3[0]: [..., ..., ..., ..., ...],
    consultas_q3[1]: [..., ..., ..., ..., ...],
    consultas_q3[2]: [..., ..., ..., ..., ...],
}

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert vetores_acervo.shape[0] == 2000 and np.allclose(np.linalg.norm(vetores_acervo, axis=1), 1, atol=1e-3), "os vetores do acervo devem estar normalizados"
_r = buscar("produto chegou atrasado", k=5)
assert list(_r.columns) == ["posicao", "semelhanca", "texto"] and len(_r) == 5 and _r["semelhanca"].is_monotonic_decreasing
_bib = util.semantic_search(modelo_frases.encode(["produto chegou atrasado"], normalize_embeddings=True), vetores_acervo, top_k=5)[0]
assert list(_r["posicao"]) == [a["corpus_id"] for a in _bib], "o resultado difere do da busca da biblioteca"
assert len(consultas_q3) == 3 and len(set(consultas_q3)) == 3
assert set(relevantes_q3) == set(consultas_q3) and all(len(v) == 5 and all(isinstance(x, bool) for x in v) for v in relevantes_q3.values())
precisao_q3 = pd.Series({consulta: np.mean(julgamento) for consulta, julgamento in relevantes_q3.items()})
print("Formato correto. Proporção de resultados relevantes entre os 5 primeiros, por consulta:")
print(precisao_q3.to_string())

**3e)** Para a consulta em que a busca foi **pior**: o que os resultados errados têm em comum com a consulta? Para a consulta em que foi **melhor**: quantos dos resultados relevantes uma busca pela palavra exata encontraria? Cite uma situação em que você preferiria a busca por palavra exata.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,5 pontos): quantos exemplos são necessários?

Vamos medir como o desempenho cresce com o número de exemplos rotulados, para duas representações do texto.

Para cada `n` em 50, 200, 1000 e 3000, usando **os primeiros `n` exemplos** de `ajuste` e avaliando sempre nos mesmos 1000 de `valid`:

- `tfidf`: AUC de TF-IDF (`min_df=1`) com regressão logística;
- `vetores`: AUC de regressão logística sobre os vetores de frases.

Monte `curva_q4`: um DataFrame com índice `n` e colunas `tfidf` e `vetores`. Calcule os vetores de `ajuste` e de `valid` **uma única vez**.

In [ ]:
# ✏️ Questão 4
# SEU CÓDIGO AQUI
curva_q4 = ...

In [ ]:
# ✅ Conferência da questão 4 (não altere)
assert list(curva_q4.index) == [50, 200, 1000, 3000] and list(curva_q4.columns) == ["tfidf", "vetores"]
assert curva_q4.stack().between(0, 1).all() and curva_q4.loc[3000, "tfidf"] > 0.9

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(curva_q4.index, curva_q4["tfidf"], marker="o", color="#6b6b6b", label="TF-IDF + regressão logística")
ax.plot(curva_q4.index, curva_q4["vetores"], marker="o", color="#2a78d6", label="vetores de frases + regressão logística")
ax.set_xscale("log"); ax.set_xticks(curva_q4.index, curva_q4.index); ax.set_xlabel("exemplos rotulados (escala logarítmica)"); ax.set_ylabel("AUC na validação")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
print("Formato correto.")
print(curva_q4.round(4).to_string())

**4b)** Descreva as duas curvas. Com quantos exemplos, se é que acontece, uma abordagem ultrapassa a outra? Proponha uma explicação para a diferença entre elas quando há **poucos** exemplos.

**4c)** Rotular uma avaliação custa tempo de uma pessoa. Com base nas curvas: se você pudesse pagar por 200 rótulos, que abordagem usaria? E com 3000? Que outros custos, além dos rótulos, entram na decisão?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,0 ponto): a página de um modelo

a) Escolha um dos três modelos desta lista, abra a página dele no Hub (o endereço é `huggingface.co/` seguido do nome) e responda: qual é a tarefa; em que idiomas e com que dados foi treinado; qual é a licença; que limitações ou vieses a página declara. Aponte **uma informação que falta** e que você gostaria de ter antes de usar o modelo em um projeto.

b) Declare como usou assistentes de IA nesta lista: em quais questões, para quê, e o que precisou corrigir.

✏️ **Sua resposta:**

_Escreva aqui._